In [1]:
pip install arxiv tqdm requests pandas --user

Note: you may need to restart the kernel to use updated packages.


In [2]:
import arxiv
import json
import time
from pathlib import Path
from tqdm.auto import tqdm

PROJECT_ROOT = Path.home() / "xai-knowledge-graph"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = RAW_DIR / "arxiv_xai_papers.json"

MAX_RESULTS = 3000

QUERY = (
    '(abs:"explainable AI" OR abs:"explainable artificial intelligence" '
    'OR abs:"interpretable machine learning" OR abs:"model interpretability" '
    'OR abs:"XAI" OR abs:"saliency map" OR abs:"feature attribution" '
    'OR abs:"counterfactual explanation" OR abs:"LIME" OR abs:"SHAP" '
    'OR abs:"Grad-CAM" OR abs:"integrated gradients" OR abs:"attention visualization") '
    'AND (cat:cs.LG OR cat:cs.AI OR cat:cs.CV)'
)
print("Query OK, length:", len(QUERY))

Query OK, length: 390


In [3]:
def fetch_arxiv_papers(query, max_results):
    client = arxiv.Client(page_size=100, delay_seconds=3.0, num_retries=5)
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )
    papers = []
    for r in tqdm(client.results(search), total=max_results, desc="Fetching"):
        try:
            papers.append({
                "arxiv_id":   r.entry_id.split("/")[-1].split("v")[0],
                "title":      r.title.strip().replace("\n", " "),
                "abstract":   r.summary.strip().replace("\n", " "),
                "authors":    [str(a) for a in r.authors],
                "year":       r.published.year,
                "published":  r.published.isoformat(),
                "categories": list(r.categories),
                "url":        r.entry_id,
                "doi":        r.doi,
            })
        except Exception as e:
            print(f" Skipped: {e}")
    return papers

start = time.time()
papers = fetch_arxiv_papers(QUERY, MAX_RESULTS)
print(f"\n✓ {len(papers)} papers in {(time.time()-start)/60:.1f} min")

Fetching:   0%|          | 0/3000 [00:00<?, ?it/s]


✓ 3000 papers in 1.6 min


In [4]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(papers, f, indent=2, ensure_ascii=False)
print(f"Saved → {OUTPUT_PATH} ({OUTPUT_PATH.stat().st_size/1024/1024:.1f} MB)")

Saved → /slurm/homes/pchandra/xai-knowledge-graph/data/raw/arxiv_xai_papers.json (5.4 MB)


In [5]:
import pandas as pd
df = pd.DataFrame(papers)
print(f"Papers: {len(df)} | Years: {df['year'].min()}–{df['year'].max()}")
print(f"Unique author names (raw): {df['authors'].explode().nunique()}")
print(f"\nSample:")
s = papers[0]
print(f"  {s['title']}\n  {s['authors'][:3]}... ({s['year']})")
print(f"  {s['abstract'][:200]}...")

Papers: 3000 | Years: 2024–2026
Unique author names (raw): 11403

Sample:
  ShaplEIG: Bayesian Experimental Design for Shapley Value Estimation
  ['David Rundel', 'Fabian Fumagalli', 'Maximilian Muschalik']... (2026)
  Shapley values are a principled attribution measure widely used in interpretable machine learning, but their exact computation scales exponentially with the number of players, motivating a wide range ...


In [6]:
# Supplemental fetch — Relevance-sorted to capture foundational XAI papers
client = arxiv.Client(page_size=100, delay_seconds=3.0, num_retries=5)
search = arxiv.Search(
    query=QUERY,
    max_results=1500,
    sort_by=arxiv.SortCriterion.Relevance,
)

relevant_papers = []
for r in tqdm(client.results(search), total=1500, desc="Relevance fetch"):
    try:
        relevant_papers.append({
            "arxiv_id":   r.entry_id.split("/")[-1].split("v")[0],
            "title":      r.title.strip().replace("\n", " "),
            "abstract":   r.summary.strip().replace("\n", " "),
            "authors":    [str(a) for a in r.authors],
            "year":       r.published.year,
            "published":  r.published.isoformat(),
            "categories": list(r.categories),
            "url":        r.entry_id,
            "doi":        r.doi,
        })
    except Exception as e:
        continue

existing_ids = {p["arxiv_id"] for p in papers}
new_papers = [p for p in relevant_papers if p["arxiv_id"] not in existing_ids]
papers = papers + new_papers
print(f"Original: {3000 - len(new_papers) + len(new_papers)}, new from relevance: {len(new_papers)}, total: {len(papers)}")

# Overwrite the raw file with the merged set
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(papers, f, indent=2, ensure_ascii=False)
print(f"Updated → {OUTPUT_PATH} ({len(papers)} papers)")

Relevance fetch:   0%|          | 0/1500 [00:00<?, ?it/s]

Original: 3000, new from relevance: 913, total: 3913
Updated → /slurm/homes/pchandra/xai-knowledge-graph/data/raw/arxiv_xai_papers.json (3913 papers)
